# Computing single-crystal elastic constants with `midas-stress`

How to invert polycrystal HEDM (or neutron / DAXM) load-stage data for single-crystal stiffness $\mathbf{C}_{\text{crystal}}$. This is the **Paper III** capability of the package, exposed through `ms.fit_single_crystal_stiffness`.

## When to use this
- The material is new / literature constants are poor or unavailable.
- You want to validate published constants against your own data.
- You need both $d_0$ and $\mathbf{C}$ from the same experiment.

## Outline
1. The inverse problem and symmetry parameterisation.
2. Cubic round-trip from one stage.
3. Noise propagation and standard errors.
4. Conditioning: why a single load is usually not enough.
5. Lower symmetries (orthorhombic, monoclinic, triclinic).
6. Confidence filtering for noisy grains.
7. Coupled $d_0$ + $\mathbf{C}$ fit from an unloaded + loaded pair.
8. Planning a multi-stage experiment — load-path design.

## 1. Setup and theory

Mechanical equilibrium at every load stage $i$ requires

$$\boldsymbol\sigma_{\text{app}}^{(i)} = \sum_g w_g\,\mathfrak U_g^\top\mathbf C_{\text{crystal}}\mathfrak U_g\{\boldsymbol\varepsilon_g^{(i)}\}$$

where $\mathfrak U_g$ is the Mandel $6\times6$ rotation of $\mathbf U_g$ and $w_g = V_g c_g / \sum V_g c_g$ are volume-confidence weights. Parameterise the stiffness by its $N_c$ independent constants

$$\mathbf C_{\text{crystal}} = \sum_{k=1}^{N_c} c_k P_k$$

(the $P_k$ are symmetry-dependent basis matrices in Mandel ordering). This turns each load stage into a $6\times N_c$ linear equation $A^{(i)}c = \sigma_{\text{app}}^{(i)}$; stacking stages gives a stacked least-squares problem for $c$.

### Independent-constant counts

| symmetry | $N_c$ | min distinct stages (generic loads) |
|---|---|---|
| cubic | 3 | 1 |
| hexagonal | 5 | 1 (well-chosen) |
| trigonal | 6 | 2 |
| tetragonal | 6 | 2 |
| orthorhombic | 9 | 2 |
| monoclinic | 13 | 3 |
| triclinic | 21 | 4 |

Each stage supplies 6 equations (the 6 components of the macroscopic stress). Under-determined systems raise `ValueError`.

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.transform import Rotation

import midas_stress as ms
from midas_stress.tensor import rotation_voigt_mandel, tensor_to_voigt, voigt_to_tensor

plt.rcParams.update({'font.size': 12, 'figure.figsize': (7, 4), 'axes.grid': True, 'grid.alpha': 0.3})

SAVE_DIR = os.path.abspath('.')

def save_fig(fig, name):
    p = os.path.join(SAVE_DIR, f'{name}.png')
    fig.savefig(p, bbox_inches='tight', dpi=120)
    print(f'saved: {p}')

def taylor_strains(C, orients, applied):
    """Uniform-strain (Taylor) per-grain strains that balance `applied` exactly."""
    M  = rotation_voigt_mandel(orients)
    Mt = np.swapaxes(M, -1, -2)
    C_lab = Mt @ C @ M
    eps_v = np.linalg.solve(C_lab.mean(axis=0), tensor_to_voigt(applied))
    return np.broadcast_to(voigt_to_tensor(eps_v), (len(orients), 3, 3)).copy()

print('midas-stress version:', ms.__version__)

midas-stress version: 0.2.3


## 2. Symmetry parameterisation — inspect the basis matrices

`symmetry_parameterisation(name)` returns the list of independent constant names and the $(N_c, 6, 6)$ stack of Mandel basis matrices. This is the building block of everything else.

In [2]:
for sym in ['cubic', 'hexagonal', 'trigonal', 'tetragonal', 'orthorhombic']:
    names, P = ms.symmetry_parameterisation(sym)
    print(f'{sym:<14} N_c={len(names)}  names={names}')

cubic          N_c=3  names=['C11', 'C12', 'C44']
hexagonal      N_c=5  names=['C11', 'C12', 'C13', 'C33', 'C44']
trigonal       N_c=6  names=['C11', 'C12', 'C13', 'C14', 'C33', 'C44']
tetragonal     N_c=6  names=['C11', 'C12', 'C13', 'C33', 'C44', 'C66']
orthorhombic   N_c=9  names=['C11', 'C22', 'C33', 'C12', 'C13', 'C23', 'C44', 'C55', 'C66']


### 2a. Round-trip a known stiffness

`stiffness_from_cij(cij_dict, sym)` rebuilds the 6×6 matrix — handy for sanity-checks and plotting. Any round-trip through `symmetry_parameterisation` must return the original matrix.

In [3]:
C_rebuilt = ms.stiffness_from_cij({'C11': 168.4, 'C12': 121.4, 'C44': 75.4}, 'cubic')
C_library = ms.get_stiffness('Cu')
print(f'max|C_rebuilt - C_library| = {np.max(np.abs(C_rebuilt - C_library)):.2e} GPa')

max|C_rebuilt - C_library| = 0.00e+00 GPa


## 3. Cubic recovery from one stage

Simplest possible case: cubic single crystal, one applied stress, random texture. One stage gives 6 equations for 3 unknowns — over-determined, and a clean round-trip confirms the machinery works.

In [4]:
cij_true = {'C11': 192.9, 'C12': 163.8, 'C44': 41.5}
C_true   = ms.stiffness_from_cij(cij_true, 'cubic')

N = 500
orient = Rotation.random(N, random_state=42).as_matrix()
vols   = np.ones(N)
applied = np.diag([0.05, 0.10, 0.15])   # mixed triaxial: 50/100/150 MPa

stage = dict(orient=orient, strain=taylor_strains(C_true, orient, applied),
             volumes=vols, applied_stress=applied, is_unloaded=False)

res = ms.fit_single_crystal_stiffness([stage], symmetry='cubic', fit_eps_iso=False)

print('fitted:       ', {k: round(v, 3) for k, v in res['cij'].items()})
print('ground truth: ', cij_true)
print(f'condition number       : {res["condition_number"]:.2f}')
print(f'well-conditioned       : {res["well_conditioned"]}')
print(f'residual ||Ac - sigma||: {res["residual_norm"]:.3e}')

fitted:        {'C11': 192.9, 'C12': 163.8, 'C44': 41.5}
ground truth:  {'C11': 192.9, 'C12': 163.8, 'C44': 41.5}
condition number       : 47.28
well-conditioned       : True
residual ||Ac - sigma||: 9.069e-17


## 4. Noise propagation

Real strains come with measurement noise (typical HEDM: ~$10^{-4}$ per component). The fit reports `cij_se` — per-constant standard errors propagated through the least-squares covariance. The SE scales linearly with noise and with $1/\sqrt{N}$ grains.

In [5]:
rng = np.random.default_rng(7)
noise_sigmas = [1e-5, 5e-5, 1e-4, 5e-4, 1e-3]
eps_taylor = taylor_strains(C_true, orient, applied)

print(f'{"noise":<10} {"C11":<18} {"C12":<18} {"C44":<18}')
for s in noise_sigmas:
    n = rng.normal(0, s, (N, 3, 3))
    n = 0.5 * (n + n.swapaxes(-1, -2))
    stage = dict(orient=orient, strain=eps_taylor + n, volumes=vols,
                 applied_stress=applied, is_unloaded=False)
    r = ms.fit_single_crystal_stiffness([stage], 'cubic', fit_eps_iso=False)
    fmt = lambda k: f'{r["cij"][k]:.3f} ± {r["cij_se"][k]:.3f}'
    print(f'{s:<10.0e} {fmt("C11"):<18} {fmt("C12"):<18} {fmt("C44"):<18}')

noise      C11                C12                C44               
1e-05      192.710 ± 0.666    164.264 ± 0.334    41.682 ± 0.336    
5e-05      187.731 ± 4.116    166.548 ± 2.062    44.217 ± 2.080    
1e-04      198.770 ± 7.723    161.935 ± 3.869    39.007 ± 3.929    
5e-04      319.538 ± 73.687   100.446 ± 36.860   -21.013 ± 35.784  
1e-03      190.994 ± 37.698   141.537 ± 19.076   32.151 ± 20.225   


### 4a. SE scales linearly with noise

In [6]:
rng = np.random.default_rng(7)
noise_sigmas = np.logspace(-5, -3, 8)
se_c11 = []
for s in noise_sigmas:
    n = rng.normal(0, s, (N, 3, 3))
    n = 0.5 * (n + n.swapaxes(-1, -2))
    stage = dict(orient=orient, strain=eps_taylor + n, volumes=vols,
                 applied_stress=applied, is_unloaded=False)
    se_c11.append(ms.fit_single_crystal_stiffness([stage], 'cubic', fit_eps_iso=False)['cij_se']['C11'])

fig, ax = plt.subplots()
ax.loglog(noise_sigmas, se_c11, 'o-', label='reported SE(C11)')
ax.loglog(noise_sigmas, noise_sigmas * se_c11[0] / noise_sigmas[0], 'k--', label='linear reference')
ax.set_xlabel('strain noise sigma')
ax.set_ylabel('SE(C11) [GPa]')
ax.set_title('noise propagation is linear')
ax.legend()
fig.tight_layout()
save_fig(fig, 'elastic_se_vs_noise')
plt.close(fig)

saved: /Users/hsharma/opt/MIDAS/packages/midas_stress/examples/elastic_se_vs_noise.png


## 5. Conditioning — why one stage often isn't enough

A uniaxial load along a symmetry axis of a single crystal is degenerate. The design matrix $A$ becomes ill-conditioned, and the reported `condition_number` blows up. Same thing happens if you choose loads that all span the same sub-space of stress space.

In [7]:
# Hexagonal Ti with a single uniaxial load along c in the crystal frame
C_ti = ms.get_stiffness('Ti')
N = 300
orient = np.broadcast_to(np.eye(3), (N, 3, 3)).copy()   # single orientation (texture-free "single crystal")
vols   = np.ones(N)
applied = np.diag([0.0, 0.0, 0.15])
stage = dict(orient=orient, strain=taylor_strains(C_ti, orient, applied),
             volumes=vols, applied_stress=applied, is_unloaded=False)

r = ms.fit_single_crystal_stiffness([stage], 'hexagonal', fit_eps_iso=False,
                                    cond_threshold=1e3)
print(f'condition number = {r["condition_number"]:.2e}')
print(f'well conditioned = {r["well_conditioned"]}  <-- flagged as not reliable')

condition number = inf
well conditioned = False  <-- flagged as not reliable


### 5a. Two diverse loads fix it

In [8]:
orient = Rotation.random(N, random_state=1).as_matrix()   # restore texture
load_library = [
    np.diag([0.05, 0.10, 0.15]),
    np.diag([0.15, 0.05, 0.10]),
    np.array([[0.0, 0.05, 0.0], [0.05, 0.0, 0.0], [0.0, 0.0, 0.10]]),
    np.array([[0.0, 0.0, 0.05], [0.0, 0.0, 0.05], [0.05, 0.05, 0.10]]),
]

for n_stages in [1, 2, 3]:
    stages = [
        dict(orient=orient, strain=taylor_strains(C_ti, orient, A),
             volumes=vols, applied_stress=A, is_unloaded=False)
        for A in load_library[:n_stages]
    ]
    r = ms.fit_single_crystal_stiffness(stages, 'hexagonal', fit_eps_iso=False)
    print(f'n_stages={n_stages}  cond={r["condition_number"]:.1f}  well-conditioned={r["well_conditioned"]}  '
          f'C11 err={abs(r["cij"]["C11"] - 162.4):.3e} GPa')

n_stages=1  cond=93.9  well-conditioned=True  C11 err=2.842e-14 GPa
n_stages=2  cond=55.4  well-conditioned=True  C11 err=1.705e-13 GPa
n_stages=3  cond=47.3  well-conditioned=True  C11 err=0.000e+00 GPa


## 6. Under-determined systems are caught early

Asking for 9 orthorhombic constants from a single stage (6 equations) raises a clear error — no silent garbage.

In [9]:
try:
    stage = dict(orient=orient, strain=np.zeros((N, 3, 3)), volumes=vols,
                 applied_stress=np.diag([0.05, 0.1, 0.15]), is_unloaded=False)
    ms.fit_single_crystal_stiffness([stage], 'orthorhombic', fit_eps_iso=False)
except ValueError as e:
    print('raised (expected):', e)

raised (expected): Under-determined: 6 equations for 9 unknowns (['C11', 'C22', 'C33', 'C12', 'C13', 'C23', 'C44', 'C55', 'C66']). Add more load stages.


## 7. Lower symmetries — orthorhombic and monoclinic

The required number of stages grows with $N_c$; each stage should stress a different subspace.

In [10]:
cij_ortho = {'C11': 320, 'C22': 200, 'C33': 330, 'C12': 72, 'C13': 70, 'C23': 68,
             'C44': 130, 'C55': 120, 'C66': 90}
C_ortho = ms.stiffness_from_cij(cij_ortho, 'orthorhombic')

N = 400
orient = Rotation.random(N, random_state=1).as_matrix()
vols   = np.ones(N)
stages = [
    dict(orient=orient, strain=taylor_strains(C_ortho, orient, A),
         volumes=vols, applied_stress=A, is_unloaded=False)
    for A in load_library
]

r = ms.fit_single_crystal_stiffness(stages, 'orthorhombic', fit_eps_iso=False)
print(f'cond={r["condition_number"]:.1f}  well-conditioned={r["well_conditioned"]}')
print('max relative error per constant:')
for k, v in cij_ortho.items():
    print(f'  {k} = {r["cij"][k]:.3f} GPa  (err {abs(r["cij"][k] - v)/v:.2e})')

cond=97.9  well-conditioned=True
max relative error per constant:
  C11 = 320.000 GPa  (err 2.26e-14)
  C22 = 200.000 GPa  (err 2.59e-14)
  C33 = 330.000 GPa  (err 9.13e-15)
  C12 = 72.000 GPa  (err 4.54e-15)
  C13 = 70.000 GPa  (err 4.26e-15)
  C23 = 68.000 GPa  (err 1.21e-14)
  C44 = 130.000 GPa  (err 3.72e-15)
  C55 = 120.000 GPa  (err 2.49e-15)
  C66 = 90.000 GPa  (err 1.67e-14)


### 7a. Monoclinic (13 constants, 4 stages)

In [11]:
cij_mono = {
    'C11': 320, 'C22': 200, 'C33': 330,
    'C44': 130, 'C55': 120, 'C66': 90,
    'C12': 72, 'C13': 70, 'C23': 68,
    'C15': 15, 'C25': 10, 'C35': 12, 'C46': 8,
}
C_mono = ms.stiffness_from_cij(cij_mono, 'monoclinic')
stages = [
    dict(orient=orient, strain=taylor_strains(C_mono, orient, A),
         volumes=vols, applied_stress=A, is_unloaded=False)
    for A in load_library
]
r = ms.fit_single_crystal_stiffness(stages, 'monoclinic', fit_eps_iso=False)
print(f'cond={r["condition_number"]:.1f}  well-conditioned={r["well_conditioned"]}')
worst = max(abs(r['cij'][k] - v) / v for k, v in cij_mono.items())
print(f'worst relative error = {worst:.2e}')

cond=212.7  well-conditioned=True
worst relative error = 2.67e-13


## 8. Confidence filtering protects the fit from bad grains

In [12]:
cij_true = {'C11': 192.9, 'C12': 163.8, 'C44': 41.5}
C_true   = ms.stiffness_from_cij(cij_true, 'cubic')
N = 400
orient = Rotation.random(N, random_state=3).as_matrix()
vols   = np.ones(N)
applied = np.diag([0.05, 0.10, 0.15])
eps_clean = taylor_strains(C_true, orient, applied)

# Corrupt 20% with big scatter, mark low-confidence
rng = np.random.default_rng(0)
bad = rng.choice(N, size=N // 5, replace=False)
eps_noisy = eps_clean.copy()
eps_noisy[bad] += rng.normal(0, 1e-2, (len(bad), 3, 3))
eps_noisy = 0.5 * (eps_noisy + eps_noisy.swapaxes(-1, -2))
conf = np.ones(N); conf[bad] = 0.1

stage_kw = dict(orient=orient, strain=eps_noisy, volumes=vols,
                applied_stress=applied, confidences=conf, is_unloaded=False)

r_nofilter = ms.fit_single_crystal_stiffness([stage_kw], 'cubic',
                                             fit_eps_iso=False, min_confidence=0.0)
r_filter   = ms.fit_single_crystal_stiffness([stage_kw], 'cubic',
                                             fit_eps_iso=False, min_confidence=0.5)

print(f'{"":<15} {"C11":<10} {"C12":<10} {"C44":<10}')
print(f'truth          {cij_true["C11"]:<10} {cij_true["C12"]:<10} {cij_true["C44"]:<10}')
for label, r in [('no filter', r_nofilter), ('min_conf=0.5', r_filter)]:
    print(f'{label:<15} {r["cij"]["C11"]:<10.2f} {r["cij"]["C12"]:<10.2f} {r["cij"]["C44"]:<10.2f}')

                C11        C12        C44       
truth          192.9      163.8      41.5      
no filter       172.29     212.25     60.35     
min_conf=0.5    204.94     157.78     36.21     


## 9. Coupled $d_0$ + $\mathbf{C}$ fit

When both the reference lattice and the elastic constants are unknown (true for any new material), provide **one unloaded stage** plus one or more loaded stages, and enable `fit_eps_iso=True`. The solver alternates:

1. Fit $\varepsilon_{\text{iso}}$ from the unloaded stage's residual (uses the current $\mathbf{C}$ guess).
2. Subtract $\varepsilon_{\text{iso}} I$ from every stage's strains.
3. Refit $\mathbf{C}$ from the corrected strains.

Converges in a handful of iterations because the two pieces are almost orthogonal.

In [13]:
cij_true = {'C11': 162.4, 'C12': 92.0, 'C13': 69.0, 'C33': 180.7, 'C44': 46.7}
C_true   = ms.stiffness_from_cij(cij_true, 'hexagonal')
N = 400
orient = Rotation.random(N, random_state=3).as_matrix()
vols   = np.ones(N)
eps_iso_inject = 300e-6   # 300 ppm isotropic d0 error

# Unloaded stage: true strain is zero, but the d0 shift makes it eps_iso * I
stage_unl = dict(
    orient=orient,
    strain=np.broadcast_to(eps_iso_inject * np.eye(3), (N, 3, 3)).copy(),
    volumes=vols,
    applied_stress=np.zeros((3, 3)),
    is_unloaded=True,
)

# Loaded stage: Taylor strain + same d0 shift
applied = np.diag([0.05, 0.10, 0.15])
stage_lod = dict(
    orient=orient,
    strain=taylor_strains(C_true, orient, applied) + eps_iso_inject * np.eye(3),
    volumes=vols,
    applied_stress=applied,
    is_unloaded=False,
)

r = ms.fit_single_crystal_stiffness([stage_unl, stage_lod], 'hexagonal',
                                    material_hint='Ti')   # used only to seed iteration

print(f'iterations           : {r["n_iter"]}')
print(f'recovered eps_iso    : {r["eps_iso"]:.3e}  (expected {eps_iso_inject:.3e})')
print('recovered constants:')
for k, v in cij_true.items():
    print(f'  {k}: {r["cij"][k]:.3f} ± {r["cij_se"][k]:.3f}  (truth {v})')

iterations           : 2
recovered eps_iso    : 3.000e-04  (expected 3.000e-04)
recovered constants:
  C11: 162.400 ± 0.000  (truth 162.4)
  C12: 92.000 ± 0.000  (truth 92.0)
  C13: 69.000 ± 0.000  (truth 69.0)
  C33: 180.700 ± 0.000  (truth 180.7)
  C44: 46.700 ± 0.000  (truth 46.7)


## 10. Full round-trip validation across every symmetry

End-to-end sanity check: pick a known stiffness, synthesize Taylor-compatible strains, fit, compare. Each row is a different crystal system.

In [14]:
cases = [
    ('cubic',        {'C11': 192.9, 'C12': 163.8, 'C44': 41.5}, 1),
    ('hexagonal',    {'C11': 162.4, 'C12': 92.0,  'C13': 69.0, 'C33': 180.7, 'C44': 46.7}, 1),
    ('trigonal',     {'C11': 497.4, 'C12': 162.7, 'C13': 115.9, 'C14': -22.0, 'C33': 501.1, 'C44': 147.4}, 2),
    ('tetragonal',   {'C11': 68, 'C12': 36, 'C13': 36, 'C33': 77, 'C44': 22, 'C66': 25}, 2),
    ('orthorhombic', {'C11': 320, 'C22': 200, 'C33': 330, 'C12': 72, 'C13': 70, 'C23': 68,
                      'C44': 130, 'C55': 120, 'C66': 90}, 3),
]

N = 400
orient = Rotation.random(N, random_state=42).as_matrix()
vols   = np.ones(N)

print(f'{"symmetry":<14} {"N_c":>4} {"worst rel err":>16} {"cond":>12}')
print('-' * 54)
for sym, cij, nstage in cases:
    C_true = ms.stiffness_from_cij(cij, sym)
    stages = [
        dict(orient=orient, strain=taylor_strains(C_true, orient, load_library[i]),
             volumes=vols, applied_stress=load_library[i], is_unloaded=False)
        for i in range(nstage)
    ]
    r = ms.fit_single_crystal_stiffness(stages, sym, fit_eps_iso=False)
    worst = max(abs(r['cij'][k] - v) / v for k, v in cij.items())
    print(f'{sym:<14} {len(cij):>4} {worst:>16.2e} {r["condition_number"]:>12.1f}')

symmetry        N_c    worst rel err         cond
------------------------------------------------------
cubic             3         2.23e-15         54.0
hexagonal         5         3.65e-15         93.8
trigonal          6         8.07e-14        163.8
tetragonal        6         6.12e-15         90.9
orthorhombic      9         1.19e-14        140.2


## 11. Load-path design — a practical recipe

How many and which loads should you plan for your experiment? Empirically:

| symmetry | recommended stages | load palette |
|---|---|---|
| cubic | 1 (any triaxial) | $\mathrm{diag}(0.5, 1.0, 1.5)\sigma_0$ |
| hexagonal | 1–2 | one triaxial + one shear |
| tetragonal / trigonal | 2 | triaxial + rotated triaxial |
| orthorhombic | 3 | 3 diagonal permutations with different axis weighting |
| monoclinic | 4 | add a shear-dominated stage |
| triclinic | 6 | three triaxial + three shears |

For HEDM, an **unloaded** stage is effectively free and lets the coupled fit recover $\varepsilon_{\text{iso}}$ (i.e. the $d_0$ error) at the same time. Always include one when possible.

## 12. Diagnostic: inspecting the stage design matrix

For debugging a fit you can build the per-stage $A^{(i)}$ directly via `build_stage_matrix`. Zero or near-zero singular values are an early warning of a load choice that cannot distinguish some constants.

In [15]:
names, P_stack = ms.symmetry_parameterisation('hexagonal')
A = ms.build_stage_matrix(
    orientations=orient,
    strains_lab=taylor_strains(C_ti, orient, load_library[0]),
    weights=vols / vols.sum(),
    P_stack=P_stack,
)
s = np.linalg.svd(A, compute_uv=False)
print(f'A shape = {A.shape}  (6 eqns, {len(names)} unknowns)')
print(f'singular values = {np.round(s, 3)}')
print(f'cond(A) = {s.max() / s.min():.2f}')

A shape = (6, 5)  (6 eqns, 5 unknowns)
singular values = [0.001 0.001 0.    0.    0.   ]
cond(A) = 93.82


## 13. Summary

- `fit_single_crystal_stiffness(stages, symmetry, ...)` is the single entry point.
- Stages are simple dicts: `orient`, `strain`, `volumes`, `applied_stress`, optionally `confidences` and `is_unloaded`.
- Output carries: fitted `cij`, `cij_se`, full covariance, condition number, residual, number of iterations, final stiffness in Mandel form.
- Combines seamlessly with `ms.compute_stress` / `ms.correct_d0` — fit $\mathbf{C}$ once, then use it for all stress analyses.
- See `crss_and_plasticity.ipynb` for downstream slip-system analysis from the per-grain stress tensors.